In [3]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from itertools import combinations
import json
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier


RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
BOOTSTRAP_N = 2000
CALIBRATION_BINS = 8
LEARNING_CURVE_POINTS = 5
TARGET = "Mortality"

DATA_PATH = Path(r"C:/Users/hosse/Desktop/Baqiyatallah Hospital/Projects/Dr. Mahdi Sotoudeh/Serum Ferritin, D-Dimer, LDH, and CRP as Predictors of Mortality in COVID-19 A Machine Learning Approach/Main.csv")
OUTPUT_DIR = Path(r"C:/Users/hosse/Desktop/Baqiyatallah Hospital/Projects/Dr. Mahdi Sotoudeh/Serum Ferritin, D-Dimer, LDH, and CRP as Predictors of Mortality in COVID-19 A Machine Learning Approach/Results_Revised")

EXCLUDE_COLUMNS = ["admcode"]

LEAKAGE_PATTERNS = [
    r"(^|[_\s])id($|[_\s])",
    r"admcode",
    r"survival.?days?",
    r"length.?of.?stay",
    r"(^|[_\s])los($|[_\s])",
    r"outcome",
    r"mechanical.?ventilat",
    r"icu.?admission",
    r"intensive.?care.?admission"
]

SHAP_BACKGROUND_SIZE = 40
SHAP_TEST_SIZE = 200
SHAP_MAX_EVALS_MULTIPLIER = 2
SHAP_MAX_DISPLAY = 20
SELECT_THRESHOLD_BY_YOUDEN = True
RUN_RISK_STRATIFIED_DCA = True

MODEL_ORDER = [
    "Logistic Regression",
    "Random Forest",
    "XGBoost",
    "SVM",
    "Voting Classifier",
    "Stacking Classifier"
]

FIG_DIR = OUTPUT_DIR / "Figures"
SHAP_DIR = FIG_DIR / "SHAP"
MODEL_DIR = OUTPUT_DIR / "Models"


def setup_output():
    for folder in [OUTPUT_DIR, FIG_DIR, SHAP_DIR, MODEL_DIR]:
        folder.mkdir(parents=True, exist_ok=True)

    plt.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 600,
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "legend.fontsize": 8
    })


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False, default=str)


def load_and_prepare_data():
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

    data = pd.read_csv(DATA_PATH)
    data.columns = [str(column).strip() for column in data.columns]

    if TARGET not in data.columns:
        raise ValueError(f"Target column '{TARGET}' was not found.")

    data = data.replace(
        [".", "", " ", "NA", "N/A", "na", "null", "None"],
        np.nan
    )

    data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")
    data = data.loc[data[TARGET].notna()].copy()
    data[TARGET] = data[TARGET].astype(int)

    if not set(data[TARGET].unique()).issubset({0, 1}):
        raise ValueError("Mortality must be coded as 0 = survived and 1 = died.")

    manual_exclusions = [
        column for column in EXCLUDE_COLUMNS if column in data.columns
    ]

    automatic_exclusions = []
    for column in data.columns:
        if column == TARGET:
            continue
        normalized = str(column).strip().lower()
        if any(
            re.search(pattern, normalized, flags=re.IGNORECASE)
            for pattern in LEAKAGE_PATTERNS
        ):
            automatic_exclusions.append(column)

    excluded = sorted(set(manual_exclusions + automatic_exclusions))
    predictors = data.drop(
        columns=[TARGET] + excluded,
        errors="ignore"
    ).copy()
    outcome = data[TARGET].copy()

    for column in predictors.columns:
        if not pd.api.types.is_numeric_dtype(predictors[column]):
            converted = pd.to_numeric(
                predictors[column],
                errors="coerce"
            )
            if converted.notna().sum() == predictors[column].notna().sum():
                predictors[column] = converted

    non_numeric = predictors.select_dtypes(
        exclude=[np.number]
    ).columns.tolist()

    if non_numeric:
        raise ValueError(
            "Non-numeric predictors remain. Define their coding explicitly: "
            + str(non_numeric)
        )

    if predictors.shape[1] == 0:
        raise ValueError("No predictor variables remain after exclusions.")

    missing = predictors.isna().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        raise ValueError(
            "Missing predictor values were found. This workflow does not "
            "perform imputation:\n" + missing.to_string()
        )

    zero_variance = predictors.columns[
        predictors.nunique(dropna=False) <= 1
    ].tolist()

    if zero_variance:
        predictors = predictors.drop(columns=zero_variance)

    if predictors.shape[1] == 0:
        raise ValueError("All predictors had zero variance.")

    predictors = predictors.astype(float)

    report = {
        "n_total": int(len(data)),
        "n_events": int(outcome.sum()),
        "mortality_rate": float(outcome.mean()),
        "n_predictors_after_exclusion": int(predictors.shape[1]),
        "predictors": predictors.columns.tolist(),
        "manual_exclusions_found": manual_exclusions,
        "automatic_leakage_exclusions": automatic_exclusions,
        "zero_variance_exclusions": zero_variance,
        "missing_predictor_values": 0,
        "imputation": "None",
        "resampling": "None"
    }

    save_json(OUTPUT_DIR / "Preprocessing_Report.json", report)

    print("\nPREPROCESSING REPORT")
    print(json.dumps(report, indent=2, ensure_ascii=False))

    return predictors, outcome, report


def split_dataset(predictors, outcome):
    x_train, x_test, y_train, y_test = train_test_split(
        predictors,
        outcome,
        test_size=TEST_SIZE,
        stratify=outcome,
        random_state=RANDOM_STATE
    )

    cv = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    print(f"\nTotal N = {len(predictors)}")
    print(f"Training N = {len(x_train)}; Test N = {len(x_test)}")
    print(f"Predictors = {predictors.shape[1]}")
    print(f"Training events = {int(y_train.sum())}")
    print(f"Test events = {int(y_test.sum())}")

    return x_train, x_test, y_train, y_test, cv


def define_model_grids(y_train):
    negative_positive_ratio = (
        (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    )

    logistic_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            random_state=RANDOM_STATE
        ))
    ])

    logistic_grid = {
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["liblinear"],
        "model__class_weight": [None, "balanced"]
    }

    random_forest_pipeline = Pipeline([
        ("model", RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ])

    random_forest_grid = {
        "model__n_estimators": [300, 600, 900],
        "model__max_depth": [None, 8, 15],
        "model__min_samples_split": [2, 10],
        "model__min_samples_leaf": [2, 5],
        "model__max_features": ["sqrt", 0.7],
        "model__class_weight": [None, "balanced"]
    }

    xgboost_pipeline = Pipeline([
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist"
        ))
    ])

    xgboost_grid = {
        "model__n_estimators": [300, 600, 900],
        "model__max_depth": [2, 3, 5],
        "model__learning_rate": [0.01, 0.03, 0.07],
        "model__subsample": [0.7, 0.9],
        "model__colsample_bytree": [0.7, 0.9],
        "model__min_child_weight": [1, 5],
        "model__reg_lambda": [1, 5, 10],
        "model__scale_pos_weight": [1.0, float(negative_positive_ratio)]
    }

    svm_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            probability=True,
            random_state=RANDOM_STATE
        ))
    ])

    svm_grid = [
        {
            "model__kernel": ["rbf"],
            "model__C": [0.1, 1, 10, 30],
            "model__gamma": ["scale", 0.01, 0.1],
            "model__class_weight": [None, "balanced"]
        },
        {
            "model__kernel": ["linear"],
            "model__C": [0.1, 1, 10, 30],
            "model__class_weight": [None, "balanced"]
        }
    ]

    return {
        "Logistic Regression": (logistic_pipeline, logistic_grid),
        "Random Forest": (random_forest_pipeline, random_forest_grid),
        "XGBoost": (xgboost_pipeline, xgboost_grid),
        "SVM": (svm_pipeline, svm_grid)
    }


def tune_base_models(x_train, y_train, cv):
    models = {}
    results = {}

    for model_name, (pipeline, parameters) in define_model_grids(y_train).items():
        print(f"\nTuning {model_name}")

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=parameters,
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1,
            refit=True,
            verbose=1,
            return_train_score=False
        )

        search.fit(x_train, y_train)

        models[model_name] = search.best_estimator_
        results[model_name] = {
            "best_parameters": search.best_params_,
            "best_cv_auc": float(search.best_score_)
        }

        pd.DataFrame(search.cv_results_).to_csv(
            OUTPUT_DIR / f"GridSearch_{model_name.replace(' ', '_')}.csv",
            index=False
        )

        print("Best CV AUC:", round(search.best_score_, 4))
        print("Best parameters:", search.best_params_)

    return models, results


def fit_ensemble_models(models, x_train, y_train, cv, results):
    base_estimators = [
        ("rf", clone(models["Random Forest"])),
        ("xgb", clone(models["XGBoost"])),
        ("svm", clone(models["SVM"]))
    ]

    voting_model = VotingClassifier(
        estimators=base_estimators,
        voting="soft",
        n_jobs=-1
    )
    voting_model.fit(x_train, y_train)
    models["Voting Classifier"] = voting_model

    stacking_model = StackingClassifier(
        estimators=[
            ("rf", clone(models["Random Forest"])),
            ("xgb", clone(models["XGBoost"])),
            ("svm", clone(models["SVM"]))
        ],
        final_estimator=LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        stack_method="predict_proba",
        cv=cv,
        passthrough=False,
        n_jobs=-1
    )
    stacking_model.fit(x_train, y_train)
    models["Stacking Classifier"] = stacking_model

    results["Voting Classifier"] = {
        "base_models": ["Random Forest", "XGBoost", "SVM"],
        "voting": "soft"
    }

    results["Stacking Classifier"] = {
        "base_models": ["Random Forest", "XGBoost", "SVM"],
        "meta_model": "Logistic Regression",
        "stacking_cv": CV_FOLDS
    }

    return models, results


def select_youden_threshold(y_true, probability):
    false_positive_rate, true_positive_rate, thresholds = roc_curve(
        y_true,
        probability
    )
    finite = np.isfinite(thresholds)
    false_positive_rate = false_positive_rate[finite]
    true_positive_rate = true_positive_rate[finite]
    thresholds = thresholds[finite]
    j_values = true_positive_rate - false_positive_rate
    index = int(np.argmax(j_values))
    return float(thresholds[index]), float(j_values[index])


def calculate_training_thresholds(models, x_train, y_train, cv, results):
    thresholds = {}
    oof_predictions = {}

    for name in MODEL_ORDER:
        print(f"Generating training OOF predictions: {name}")

        probability = cross_val_predict(
            clone(models[name]),
            x_train,
            y_train,
            cv=cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        oof_predictions[name] = probability

        if SELECT_THRESHOLD_BY_YOUDEN:
            threshold, j_value = select_youden_threshold(
                y_train,
                probability
            )
        else:
            threshold, j_value = 0.50, np.nan

        thresholds[name] = threshold
        results[name]["training_oof_threshold"] = threshold
        results[name]["training_oof_youden_j"] = j_value

    return thresholds, oof_predictions


def bootstrap_auc_ci(y_true, probability, n_bootstrap=BOOTSTRAP_N):
    y_true = np.asarray(y_true)
    probability = np.asarray(probability)
    rng = np.random.default_rng(RANDOM_STATE)
    values = []

    for _ in range(n_bootstrap):
        indices = rng.integers(0, len(y_true), len(y_true))
        if np.unique(y_true[indices]).size == 2:
            values.append(
                roc_auc_score(y_true[indices], probability[indices])
            )

    auc = roc_auc_score(y_true, probability)

    if len(values) < 100:
        return float(auc), np.nan, np.nan

    lower, upper = np.percentile(values, [2.5, 97.5])
    return float(auc), float(lower), float(upper)


def calibration_intercept_slope(y_true, probability):
    y_array = np.asarray(y_true).astype(int)
    clipped = np.clip(np.asarray(probability), 1e-6, 1 - 1e-6)
    logit_probability = np.log(
        clipped / (1 - clipped)
    ).reshape(-1, 1)

    calibration_model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=3000
    )
    calibration_model.fit(logit_probability, y_array)

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0, 0])
    )


def evaluate_models(models, thresholds, x_test, y_test):
    probabilities = {}
    predictions = {}
    roc_data = {}
    calibration_data = {}
    rows = []

    test_predictions = pd.DataFrame({
        "Observed_Mortality": y_test.to_numpy()
    })

    for name in MODEL_ORDER:
        probability = models[name].predict_proba(x_test)[:, 1]
        prediction = (probability >= thresholds[name]).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            prediction,
            labels=[0, 1]
        ).ravel()

        auc, auc_low, auc_high = bootstrap_auc_ci(
            y_test,
            probability
        )

        calibration_intercept, calibration_slope = calibration_intercept_slope(
            y_test,
            probability
        )

        rows.append({
            "Model": name,
            "Threshold": thresholds[name],
            "AUC": auc,
            "AUC_95CI_Lower": auc_low,
            "AUC_95CI_Upper": auc_high,
            "PR_AUC": average_precision_score(y_test, probability),
            "Brier_Score": brier_score_loss(y_test, probability),
            "Calibration_Intercept": calibration_intercept,
            "Calibration_Slope": calibration_slope,
            "Accuracy": accuracy_score(y_test, prediction),
            "Balanced_Accuracy": balanced_accuracy_score(y_test, prediction),
            "Sensitivity": recall_score(y_test, prediction, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "Precision_PPV": precision_score(y_test, prediction, zero_division=0),
            "NPV": tn / (tn + fn) if (tn + fn) else np.nan,
            "F1": f1_score(y_test, prediction, zero_division=0),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp
        })

        probabilities[name] = probability
        predictions[name] = prediction
        roc_data[name] = roc_curve(y_test, probability)[:2]
        calibration_data[name] = calibration_curve(
            y_test,
            probability,
            n_bins=CALIBRATION_BINS,
            strategy="quantile"
        )

        test_predictions[f"{name}_Probability"] = probability
        test_predictions[f"{name}_Prediction"] = prediction

    performance = pd.DataFrame(rows).sort_values(
        "AUC",
        ascending=False
    )

    performance.to_csv(
        OUTPUT_DIR / "Model_Performance_Results.csv",
        index=False
    )

    test_predictions.to_csv(
        OUTPUT_DIR / "Test_Set_Predictions.csv",
        index=False
    )

    performance.to_excel(
        OUTPUT_DIR / "Final_Model_Performance.xlsx",
        index=False
    )

    print("\nINDEPENDENT TEST-SET RESULTS")
    print(performance.round(4).to_string(index=False))

    return performance, probabilities, predictions, roc_data, calibration_data


def plot_model_evaluation(axes, name, model, x_train, y_train, y_test, cv,
                          performance, probabilities, roc_data,
                          calibration_data):
    probability = probabilities[name]
    false_positive_rate, true_positive_rate = roc_data[name]

    auc_value = performance.loc[
        performance["Model"] == name,
        "AUC"
    ].iloc[0]

    axes[0].plot(
        false_positive_rate,
        true_positive_rate,
        lw=2,
        label=f"AUC = {auc_value:.3f}"
    )
    axes[0].plot([0, 1], [0, 1], "--", color="gray", lw=1)
    axes[0].set(
        title="ROC Curve",
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        xlim=(0, 1),
        ylim=(0, 1)
    )
    axes[0].legend(loc="lower right")

    y_array = np.asarray(y_test)
    axes[1].hist(
        probability[y_array == 0],
        bins=18,
        alpha=0.55,
        label="Survived"
    )
    axes[1].hist(
        probability[y_array == 1],
        bins=18,
        alpha=0.55,
        label="Died"
    )
    axes[1].set(
        title="Predicted Probability Distribution",
        xlabel="Predicted probability of mortality",
        ylabel="Number of patients"
    )
    axes[1].legend()

    try:
        train_sizes, train_scores, validation_scores = learning_curve(
            clone(model),
            x_train,
            y_train,
            cv=cv,
            train_sizes=np.linspace(
                0.25,
                1.0,
                LEARNING_CURVE_POINTS
            ),
            scoring="roc_auc",
            n_jobs=-1,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        train_mean = np.nanmean(train_scores, axis=1)
        train_sd = np.nanstd(train_scores, axis=1)
        validation_mean = np.nanmean(validation_scores, axis=1)
        validation_sd = np.nanstd(validation_scores, axis=1)

        axes[2].plot(
            train_sizes,
            train_mean,
            "o-",
            label="Training AUC"
        )
        axes[2].plot(
            train_sizes,
            validation_mean,
            "o-",
            label="Validation AUC"
        )
        axes[2].fill_between(
            train_sizes,
            train_mean - train_sd,
            train_mean + train_sd,
            alpha=0.15
        )
        axes[2].fill_between(
            train_sizes,
            validation_mean - validation_sd,
            validation_mean + validation_sd,
            alpha=0.15
        )
        axes[2].set(
            title="Learning Curve",
            xlabel="Number of training patients",
            ylabel="ROC-AUC",
            ylim=(0.4, 1.02)
        )
        axes[2].legend()

    except Exception as error:
        axes[2].text(
            0.05,
            0.5,
            f"Learning curve unavailable:\n{error}",
            transform=axes[2].transAxes
        )
        axes[2].set_title("Learning Curve", loc="center")

    mean_predicted, observed_fraction = calibration_data[name]
    brier = performance.loc[
        performance["Model"] == name,
        "Brier_Score"
    ].iloc[0]

    axes[3].plot(
        [0, 1],
        [0, 1],
        "--",
        color="gray",
        label="Ideal"
    )
    axes[3].plot(
        mean_predicted,
        observed_fraction,
        "o-",
        lw=2,
        label="Model"
    )
    axes[3].set(
        title=f"Calibration Plot (Brier={brier:.3f})",
        xlabel="Mean predicted probability",
        ylabel="Observed event frequency",
        xlim=(0, 1),
        ylim=(0, 1)
    )
    axes[3].legend()

    for axis in axes:
        axis.title.set_ha("center")
        axis.grid(alpha=0.2)


def create_evaluation_figures(models, x_train, y_train, y_test, cv,
                              performance, probabilities, roc_data,
                              calibration_data):
    for name in MODEL_ORDER:
        figure, axes = plt.subplots(2, 2, figsize=(13, 10))

        plot_model_evaluation(
            axes.ravel(),
            name,
            models[name],
            x_train,
            y_train,
            y_test,
            cv,
            performance,
            probabilities,
            roc_data,
            calibration_data
        )

        figure.suptitle(
            f"Model Evaluation Summary — {name}",
            fontsize=16,
            y=1.01,
            ha="center"
        )
        figure.tight_layout()
        figure.savefig(
            FIG_DIR / f"Evaluation_{name.replace(' ', '_')}_600dpi.png",
            dpi=600,
            bbox_inches="tight"
        )
        plt.close(figure)

    figure = plt.figure(figsize=(24, 16))
    outer_grid = figure.add_gridspec(
        2,
        3,
        left=0.045,
        right=0.99,
        bottom=0.04,
        top=0.91,
        wspace=0.28,
        hspace=0.34
    )

    for model_index, name in enumerate(MODEL_ORDER):
        model_row = model_index // 3
        model_column = model_index % 3

        inner_grid = outer_grid[
            model_row,
            model_column
        ].subgridspec(
            2,
            2,
            wspace=0.32,
            hspace=0.38
        )

        model_axes = [
            figure.add_subplot(inner_grid[0, 0]),
            figure.add_subplot(inner_grid[0, 1]),
            figure.add_subplot(inner_grid[1, 0]),
            figure.add_subplot(inner_grid[1, 1])
        ]

        plot_model_evaluation(
            model_axes,
            name,
            models[name],
            x_train,
            y_train,
            y_test,
            cv,
            performance,
            probabilities,
            roc_data,
            calibration_data
        )

        model_axes[0].text(
            0.5,
            1.28,
            name,
            transform=model_axes[0].transAxes,
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold"
        )

    figure.suptitle(
        "Models Evaluation Summary",
        fontsize=20,
        y=0.98,
        ha="center"
    )
    figure.savefig(
        FIG_DIR / "Evaluation_All_Models_Combined_600dpi.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(figure)


def create_comparative_figures(y_test, probabilities, roc_data,
                               calibration_data, performance):
    figure, axis = plt.subplots(figsize=(9, 8))

    for name in MODEL_ORDER:
        false_positive_rate, true_positive_rate = roc_data[name]
        auc_value = performance.loc[
            performance["Model"] == name,
            "AUC"
        ].iloc[0]

        axis.plot(
            false_positive_rate,
            true_positive_rate,
            lw=2,
            label=f"{name} (AUC={auc_value:.3f})"
        )

    axis.plot([0, 1], [0, 1], "--", color="gray", label="Chance")
    axis.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title="ROC Curves — Independent Test Set"
    )
    axis.title.set_ha("center")
    axis.legend(fontsize=8)
    figure.tight_layout()
    figure.savefig(
        FIG_DIR / "ROC_All_Models_600dpi.png",
        bbox_inches="tight"
    )
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(9, 8))
    prevalence = float(np.mean(y_test))
    axis.axhline(
        prevalence,
        linestyle="--",
        color="gray",
        label=f"No-skill prevalence = {prevalence:.3f}"
    )

    for name in MODEL_ORDER:
        precision, recall, _ = precision_recall_curve(
            y_test,
            probabilities[name]
        )
        pr_auc = performance.loc[
            performance["Model"] == name,
            "PR_AUC"
        ].iloc[0]
        axis.plot(
            recall,
            precision,
            lw=2,
            label=f"{name} (PR-AUC={pr_auc:.3f})"
        )

    axis.set(
        xlabel="Recall (Sensitivity)",
        ylabel="Precision (PPV)",
        title="Precision–Recall Curves — Independent Test Set"
    )
    axis.title.set_ha("center")
    axis.legend(fontsize=8)
    figure.tight_layout()
    figure.savefig(
        FIG_DIR / "PR_All_Models_600dpi.png",
        bbox_inches="tight"
    )
    plt.close(figure)

    figure, axis = plt.subplots(figsize=(9, 8))
    axis.plot([0, 1], [0, 1], "--", color="gray", label="Ideal")

    for name in MODEL_ORDER:
        mean_predicted, observed = calibration_data[name]
        axis.plot(
            mean_predicted,
            observed,
            "o-",
            lw=1.8,
            label=name
        )

    axis.set(
        xlim=(0, 1),
        ylim=(0, 1),
        xlabel="Mean predicted probability",
        ylabel="Observed event frequency",
        title="Calibration Curves — Independent Test Set"
    )
    axis.title.set_ha("center")
    axis.legend(fontsize=8)
    figure.tight_layout()
    figure.savefig(
        FIG_DIR / "Calibration_All_Models_600dpi.png",
        bbox_inches="tight"
    )
    plt.close(figure)


def paired_auc_bootstrap(y_true, probability_a, probability_b):
    y_array = np.asarray(y_true)
    probability_a = np.asarray(probability_a)
    probability_b = np.asarray(probability_b)
    rng = np.random.default_rng(RANDOM_STATE)

    observed = (
        roc_auc_score(y_array, probability_a)
        - roc_auc_score(y_array, probability_b)
    )

    differences = []

    for _ in range(BOOTSTRAP_N):
        indices = rng.integers(0, len(y_array), len(y_array))
        if np.unique(y_array[indices]).size < 2:
            continue

        differences.append(
            roc_auc_score(y_array[indices], probability_a[indices])
            - roc_auc_score(y_array[indices], probability_b[indices])
        )

    if len(differences) < 100:
        return float(observed), np.nan, np.nan, np.nan

    lower, upper = np.percentile(differences, [2.5, 97.5])
    differences = np.asarray(differences)

    p_value = 2 * min(
        np.mean(differences <= 0),
        np.mean(differences >= 0)
    )

    return (
        float(observed),
        float(lower),
        float(upper),
        float(min(1, p_value))
    )


def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(p_values.shape, np.nan, dtype=float)
    valid_indices = np.where(np.isfinite(p_values))[0]

    if len(valid_indices) == 0:
        return adjusted

    valid_p = p_values[valid_indices]
    order = np.argsort(valid_p)
    ranked_p = valid_p[order]
    count = len(ranked_p)

    ranked_adjusted = ranked_p * count / np.arange(1, count + 1)
    ranked_adjusted = np.minimum.accumulate(ranked_adjusted[::-1])[::-1]
    ranked_adjusted = np.clip(ranked_adjusted, 0, 1)

    original_order_adjusted = np.empty(count, dtype=float)
    original_order_adjusted[order] = ranked_adjusted
    adjusted[valid_indices] = original_order_adjusted

    return adjusted


def compare_models_auc(y_test, probabilities):
    rows = []

    for model_a, model_b in combinations(MODEL_ORDER, 2):
        difference, lower, upper, p_value = paired_auc_bootstrap(
            y_test,
            probabilities[model_a],
            probabilities[model_b]
        )

        rows.append({
            "Model A": model_a,
            "Model B": model_b,
            "AUC difference (A-B)": difference,
            "CI lower": lower,
            "CI upper": upper,
            "P value": p_value
        })

    pairwise = pd.DataFrame(rows)
    pairwise["FDR_adjusted_P"] = benjamini_hochberg(
        pairwise["P value"].to_numpy()
    )

    pairwise.to_csv(
        OUTPUT_DIR / "Pairwise_AUC_Bootstrap_Comparisons.csv",
        index=False
    )


def calculate_net_benefit(y_true, probability, threshold):
    y_array = np.asarray(y_true).astype(int)
    probability = np.asarray(probability)
    predicted_positive = probability >= threshold

    true_positive = np.sum(predicted_positive & (y_array == 1))
    false_positive = np.sum(predicted_positive & (y_array == 0))

    return (
        true_positive / len(y_array)
        - false_positive / len(y_array) * threshold / (1 - threshold)
    )


def create_standard_dca(y_test, probabilities):
    thresholds = np.arange(0.01, 0.501, 0.01)
    results = {"Threshold": thresholds}

    for name in MODEL_ORDER:
        results[name] = [
            calculate_net_benefit(
                y_test,
                probabilities[name],
                threshold
            )
            for threshold in thresholds
        ]

    prevalence = float(np.mean(y_test))
    results["Treat All"] = [
        prevalence - (1 - prevalence) * threshold / (1 - threshold)
        for threshold in thresholds
    ]
    results["Treat None"] = np.zeros(len(thresholds))

    dca = pd.DataFrame(results)
    dca.to_csv(
        OUTPUT_DIR / "Decision_Curve_Analysis.csv",
        index=False
    )

    figure, axis = plt.subplots(figsize=(10, 8))

    for name in MODEL_ORDER:
        axis.plot(
            thresholds,
            dca[name],
            lw=2,
            label=name
        )

    axis.plot(
        thresholds,
        dca["Treat All"],
        "--",
        color="black",
        label="Treat All"
    )
    axis.plot(
        thresholds,
        dca["Treat None"],
        ":",
        color="black",
        label="Treat None"
    )
    axis.axhline(0, color="gray", lw=0.8)
    axis.set(
        xlim=(0.01, 0.50),
        xlabel="Threshold probability",
        ylabel="Net benefit",
        title="Decision Curve Analysis — Independent Test Set"
    )
    axis.title.set_ha("center")
    axis.legend(fontsize=8)
    axis.grid(alpha=0.2)
    figure.tight_layout()
    figure.savefig(
        FIG_DIR / "DCA_All_Models_600dpi.png",
        bbox_inches="tight"
    )
    plt.close(figure)


def calculate_shap_explanations(models, x_train, x_test):
    rng = np.random.default_rng(RANDOM_STATE)

    background_count = min(SHAP_BACKGROUND_SIZE, len(x_train))
    explain_count = min(SHAP_TEST_SIZE, len(x_test))

    background = x_train.iloc[
        rng.choice(len(x_train), background_count, replace=False)
    ].copy()

    explain_data = x_test.iloc[
        rng.choice(len(x_test), explain_count, replace=False)
    ].copy()

    explanations = {}

    for name in MODEL_ORDER:
        print(f"\nCalculating SHAP for {name}")

        def predict_probability(values, fitted_model=models[name]):
            if isinstance(values, pd.DataFrame):
                frame = values.loc[:, x_train.columns]
            else:
                frame = pd.DataFrame(
                    values,
                    columns=x_train.columns
                )
            return fitted_model.predict_proba(frame)[:, 1]

        explainer = shap.Explainer(
            predict_probability,
            background,
            algorithm="permutation",
            feature_names=x_train.columns.tolist(),
            seed=RANDOM_STATE
        )

        max_evaluations = max(
            SHAP_MAX_EVALS_MULTIPLIER * x_train.shape[1] + 1,
            2 * x_train.shape[1] + 1
        )

        explanations[name] = explainer(
            explain_data,
            max_evals=max_evaluations
        )

        np.save(
            SHAP_DIR / f"{name.replace(' ', '_')}_SHAP_values.npy",
            np.asarray(explanations[name].values)
        )

    return explanations, explain_data


def create_shap_beeswarm(explanations, number_of_features):
    figure, axes = plt.subplots(
        2,
        3,
        figsize=(25, 16)
    )

    for index, name in enumerate(MODEL_ORDER):
        shap.plots.beeswarm(
            explanations[name],
            max_display=min(SHAP_MAX_DISPLAY, number_of_features),
            ax=axes.ravel()[index],
            show=False,
            color_bar=(index == 2),
            plot_size=None
        )

        axes.ravel()[index].set_title(
            name,
            fontweight="bold",
            loc="center",
            x=0.5
        )
        axes.ravel()[index].title.set_horizontalalignment("center")

    figure.suptitle(
        "SHAP Summary Plots — All Models",
        fontsize=18,
        ha="center",
        x=0.5
    )
    figure.subplots_adjust(top=0.91, hspace=0.38, wspace=0.32)
    figure.savefig(
        SHAP_DIR / "SHAP_Summary_All_Models_600dpi.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(figure)


def choose_primary_shap_model(results, oof_predictions, y_train):
    cv_auc = {
        name: results[name]["best_cv_auc"]
        for name in [
            "Logistic Regression",
            "Random Forest",
            "XGBoost",
            "SVM"
        ]
    }

    for name in ["Voting Classifier", "Stacking Classifier"]:
        cv_auc[name] = float(
            roc_auc_score(y_train, oof_predictions[name])
        )

    primary_name = max(cv_auc, key=cv_auc.get)
    return primary_name, cv_auc


def create_primary_shap_outputs(primary_name, explanations, explain_data,
                                models):
    primary_explanation = explanations[primary_name]
    shap_values = np.asarray(primary_explanation.values)

    importance = pd.DataFrame({
        "Feature": explain_data.columns,
        "Mean_Absolute_SHAP": np.mean(np.abs(shap_values), axis=0)
    }).sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    )

    importance.to_csv(
        SHAP_DIR / "Primary_Model_SHAP_Global_Importance.csv",
        index=False
    )

    probabilities = models[primary_name].predict_proba(
        explain_data
    )[:, 1]

    highest_risk_index = int(np.argmax(probabilities))

    shap.plots.waterfall(
        primary_explanation[highest_risk_index],
        max_display=min(15, explain_data.shape[1]),
        show=False
    )
    figure = plt.gcf()
    figure.tight_layout()
    figure.savefig(
        SHAP_DIR / "SHAP_Waterfall_Highest_Risk_Test_Patient_600dpi.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(figure)

    return primary_explanation, probabilities, importance


def get_scalar_base_value(explanation, row_index):
    base_values = np.asarray(explanation.base_values)

    if base_values.ndim == 0:
        return float(base_values)

    if base_values.ndim == 1:
        return float(base_values[row_index])

    return float(base_values[row_index].reshape(-1)[0])


def create_one_model_decision_plot(model_name, explanation, explain_data,
                                   probability):
    risk_indices = {
        "Low-risk patient": int(np.argmin(probability)),
        "High-risk patient": int(np.argmax(probability))
    }

    figure, axes = plt.subplots(
        1,
        2,
        figsize=(22, 9)
    )

    shap_values = np.asarray(explanation.values)
    selected_rows = []

    for axis, (risk_label, row_index) in zip(
        axes,
        risk_indices.items()
    ):
        plt.sca(axis)

        shap.decision_plot(
            get_scalar_base_value(explanation, row_index),
            shap_values[row_index],
            features=explain_data.iloc[row_index].to_numpy(),
            feature_names=explain_data.columns.tolist(),
            feature_order="importance",
            feature_display_range=slice(
                -min(12, explain_data.shape[1]),
                None
            ),
            show=False,
            ignore_warnings=True
        )

        axis.set_title(
            f"{model_name} — {risk_label}",
            loc="center",
            x=0.5,
            fontweight="bold"
        )
        axis.title.set_horizontalalignment("center")
        axis.tick_params(axis="y", labelsize=10)
        axis.tick_params(axis="x", labelsize=10)
        axis.set_xlabel("Predicted mortality probability")
        axis.grid(alpha=0.2)

        selected_rows.append({
            "Model": model_name,
            "Risk_Group": risk_label,
            "Test_Row_Index": explain_data.index[row_index],
            "Predicted_Mortality_Probability": float(
                probability[row_index]
            )
        })

    figure.suptitle(
        f"SHAP Decision Plot — {model_name}",
        fontsize=17,
        ha="center",
        x=0.5
    )
    figure.subplots_adjust(top=0.86, wspace=0.42, bottom=0.16)
    figure.savefig(
        SHAP_DIR / f"SHAP_Decision_{model_name.replace(' ', '_')}_600dpi.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(figure)

    return selected_rows


def create_all_model_decision_plots(explanations, explain_data, models):
    figure, axes = plt.subplots(
        len(MODEL_ORDER),
        2,
        figsize=(25, 7.2 * len(MODEL_ORDER)),
        squeeze=False
    )

    selected_patients = []

    for row, model_name in enumerate(MODEL_ORDER):
        probability = models[model_name].predict_proba(
            explain_data
        )[:, 1]

        explanation = explanations[model_name]
        shap_values = np.asarray(explanation.values)

        selections = {
            "Low-risk patient": int(np.argmin(probability)),
            "High-risk patient": int(np.argmax(probability))
        }

        individual_rows = create_one_model_decision_plot(
            model_name,
            explanation,
            explain_data,
            probability
        )
        selected_patients.extend(individual_rows)

        for column, (risk_label, row_index) in enumerate(selections.items()):
            axis = axes[row, column]
            plt.sca(axis)

            shap.decision_plot(
                get_scalar_base_value(explanation, row_index),
                shap_values[row_index],
                features=explain_data.iloc[row_index].to_numpy(),
                feature_names=explain_data.columns.tolist(),
                feature_order="importance",
                feature_display_range=slice(
                    -min(10, explain_data.shape[1]),
                    None
                ),
                show=False,
                ignore_warnings=True
            )

            axis.set_title(
                f"{model_name} — {risk_label}",
                loc="center",
                x=0.5,
                fontweight="bold",
                pad=12
            )
            axis.title.set_horizontalalignment("center")
            axis.set_xlabel("Predicted mortality probability", fontsize=11)
            axis.tick_params(axis="y", labelsize=9)
            axis.tick_params(axis="x", labelsize=10)
            axis.grid(alpha=0.2)

    figure.suptitle(
        "SHAP Decision Plots — Low-risk and High-risk Patients Across All Models",
        fontsize=20,
        ha="center",
        x=0.5,
        y=0.995
    )
    figure.subplots_adjust(
        top=0.975,
        bottom=0.035,
        left=0.12,
        right=0.985,
        hspace=0.72,
        wspace=0.42
    )
    figure.savefig(
        SHAP_DIR / "SHAP_Decision_Low_High_Risk_All_Models_600dpi.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(figure)

    pd.DataFrame(selected_patients).to_csv(
        SHAP_DIR / "SHAP_Decision_Patient_Selection.csv",
        index=False
    )


def create_shap_risk_stratified_dca(primary_name, primary_explanation,
                                    explain_data, y_test, models):
    shap_values = np.asarray(primary_explanation.values)
    base_values = np.asarray(primary_explanation.base_values)

    if base_values.ndim == 0:
        base_values = np.repeat(base_values, len(explain_data))
    elif base_values.ndim > 1:
        base_values = base_values.reshape(
            len(explain_data),
            -1
        )[:, 0]

    shap_model_output = base_values + shap_values.sum(axis=1)
    lower_cut, upper_cut = np.quantile(
        shap_model_output,
        [1 / 3, 2 / 3]
    )

    group_masks = {
        "Low SHAP-risk group": shap_model_output <= lower_cut,
        "High SHAP-risk group": shap_model_output >= upper_cut
    }

    actual_outcome = y_test.loc[explain_data.index].to_numpy().astype(int)
    predicted_probability = models[primary_name].predict_proba(
        explain_data
    )[:, 1]

    thresholds = np.arange(0.01, 0.501, 0.01)
    dca_results = {"Threshold": thresholds}
    group_characteristics = []

    for group_name, mask in group_masks.items():
        group_size = int(mask.sum())

        if group_size < 10:
            print(
                f"Risk-stratified DCA skipped for {group_name}: "
                "fewer than 10 patients."
            )
            continue

        group_y = actual_outcome[mask]
        group_probability = predicted_probability[mask]
        group_prevalence = float(group_y.mean())

        group_characteristics.append({
            "Risk_Group": group_name,
            "N": group_size,
            "Deaths": int(group_y.sum()),
            "Observed_Mortality_Rate": group_prevalence,
            "Mean_Predicted_Risk": float(group_probability.mean())
        })

        dca_results[group_name] = [
            calculate_net_benefit(
                group_y,
                group_probability,
                threshold
            )
            for threshold in thresholds
        ]

        dca_results[f"{group_name} - Treat All"] = [
            group_prevalence
            - (1 - group_prevalence) * threshold / (1 - threshold)
            for threshold in thresholds
        ]

        dca_results[f"{group_name} - Treat None"] = np.zeros(
            len(thresholds)
        )

    if not group_characteristics:
        print("Risk-stratified DCA was not generated.")
        return

    pd.DataFrame(group_characteristics).to_csv(
        OUTPUT_DIR / "SHAP_Risk_Group_Characteristics.csv",
        index=False
    )

    dca_frame = pd.DataFrame(dca_results)
    dca_frame.to_csv(
        OUTPUT_DIR / "SHAP_Risk_Stratified_DCA.csv",
        index=False
    )

    figure, axis = plt.subplots(figsize=(10, 8))

    for column in dca_frame.columns:
        if column == "Threshold":
            continue

        line_style = "--" if "Treat All" in column else "-"
        axis.plot(
            thresholds,
            dca_frame[column],
            lw=2,
            linestyle=line_style,
            label=column
        )

    axis.axhline(0, color="gray", lw=0.8)
    axis.set(
        xlabel="Threshold probability",
        ylabel="Net benefit",
        title=(
            "SHAP-informed Risk-stratified DCA\n"
            f"Primary model: {primary_name}"
        ),
        xlim=(0.01, 0.50)
    )
    axis.title.set_ha("center")
    axis.legend(fontsize=8)
    axis.grid(alpha=0.2)
    figure.tight_layout()
    figure.savefig(
        FIG_DIR / "SHAP_Risk_Stratified_DCA_600dpi.png",
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(figure)


def save_final_outputs(models, results, thresholds, performance,
                       primary_name, cv_auc):
    for name, model in models.items():
        joblib.dump(
            model,
            MODEL_DIR / f"{name.replace(' ', '_')}.joblib"
        )

    for name in MODEL_ORDER:
        results[name]["training_oof_threshold"] = thresholds[name]
        results[name]["cv_auc_for_shap_selection"] = cv_auc[name]

    results["primary_shap_model"] = primary_name

    save_json(
        OUTPUT_DIR / "Model_Parameters_and_Thresholds.json",
        results
    )

    parameter_table = pd.DataFrame([
        {
            "Model": name,
            "Best_CV_AUC": cv_auc[name],
            "Threshold": thresholds[name],
            "Parameters": json.dumps(
                results[name],
                ensure_ascii=False,
                default=str
            )
        }
        for name in MODEL_ORDER
    ])

    parameter_table.to_excel(
        OUTPUT_DIR / "Final_Model_Parameters_and_Thresholds.xlsx",
        index=False
    )

    readme = (
        "COVID-19 mortality prediction analysis\n"
        "No imputation or resampling was used.\n"
        "All eligible numeric predictors were retained; no biomarker-specific "
        "selection or pathophysiological feature grouping was performed.\n"
        "The independent test set was reserved for final evaluation.\n"
        "The learning curve was used instead of epoch-based loss plots because "
        "the models are classical machine-learning algorithms.\n"
        "SHAP decision plots show the lowest-risk and highest-risk patients "
        "within the SHAP explanation sample for each model.\n"
        "SHAP-informed risk-stratified DCA is exploratory and is not a validated "
        "clinical risk classification rule.\n"
    )

    (OUTPUT_DIR / "Analysis_Readme.txt").write_text(
        readme,
        encoding="utf-8"
    )


def run_pipeline():
    setup_output()

    predictors, outcome, preprocessing_report = load_and_prepare_data()

    x_train, x_test, y_train, y_test, cv = split_dataset(
        predictors,
        outcome
    )

    models, results = tune_base_models(
        x_train,
        y_train,
        cv
    )

    models, results = fit_ensemble_models(
        models,
        x_train,
        y_train,
        cv,
        results
    )

    thresholds, oof_predictions = calculate_training_thresholds(
        models,
        x_train,
        y_train,
        cv,
        results
    )

    performance, probabilities, predictions, roc_data, calibration_data = (
        evaluate_models(
            models,
            thresholds,
            x_test,
            y_test
        )
    )

    create_evaluation_figures(
        models,
        x_train,
        y_train,
        y_test,
        cv,
        performance,
        probabilities,
        roc_data,
        calibration_data
    )

    create_comparative_figures(
        y_test,
        probabilities,
        roc_data,
        calibration_data,
        performance
    )

    compare_models_auc(
        y_test,
        probabilities
    )

    create_standard_dca(
        y_test,
        probabilities
    )

    explanations, explain_data = calculate_shap_explanations(
        models,
        x_train,
        x_test
    )

    create_shap_beeswarm(
        explanations,
        predictors.shape[1]
    )

    primary_name, cv_auc = choose_primary_shap_model(
        results,
        oof_predictions,
        y_train
    )

    primary_explanation, primary_probabilities, shap_importance = (
        create_primary_shap_outputs(
            primary_name,
            explanations,
            explain_data,
            models
        )
    )

    create_all_model_decision_plots(
        explanations,
        explain_data,
        models
    )

    if RUN_RISK_STRATIFIED_DCA:
        create_shap_risk_stratified_dca(
            primary_name,
            primary_explanation,
            explain_data,
            y_test,
            models
        )

    save_final_outputs(
        models,
        results,
        thresholds,
        performance,
        primary_name,
        cv_auc
    )

    print("\nANALYSIS COMPLETED")
    print("Primary SHAP model:", primary_name)
    print("Results saved to:", OUTPUT_DIR)
    print("Figures saved to:", FIG_DIR)
    print("SHAP outputs saved to:", SHAP_DIR)
    print("Models saved to:", MODEL_DIR)


if __name__ == "__main__":
    run_pipeline()


PREPROCESSING REPORT
{
  "n_total": 4734,
  "n_events": 618,
  "mortality_rate": 0.13054499366286437,
  "n_predictors_after_exclusion": 29,
  "predictors": [
    "Sex",
    "Age",
    "Marital",
    "Employment",
    "Hypertension",
    "Diabetes",
    "BUN",
    "Creatinine",
    "AST",
    "ALT",
    "ALP",
    "LDH",
    "Na",
    "K",
    "CRP",
    "D_dimer",
    "Ferritin",
    "ESR",
    "PT",
    "PTT",
    "RBC",
    "WBC",
    "HB",
    "Hct",
    "Platelets",
    "MPV",
    "MCH",
    "RDW_CV",
    "Troponin"
  ],
  "manual_exclusions_found": [
    "admcode"
  ],
  "automatic_leakage_exclusions": [
    "admcode"
  ],
  "zero_variance_exclusions": [],
  "missing_predictor_values": 0,
  "imputation": "None",
  "resampling": "None"
}

Total N = 4734
Training N = 3787; Test N = 947
Predictors = 29
Training events = 494
Test events = 124

Tuning Logistic Regression
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best CV AUC: 0.844
Best parameters: {'model__C': 10, 'm

PermutationExplainer explainer: 201it [00:40,  3.59it/s]                         



Calculating SHAP for XGBoost

Calculating SHAP for SVM


PermutationExplainer explainer: 201it [00:27,  4.74it/s]                         



Calculating SHAP for Voting Classifier


PermutationExplainer explainer: 201it [01:20,  2.22it/s]                         



Calculating SHAP for Stacking Classifier


PermutationExplainer explainer: 201it [01:19,  2.22it/s]                         



ANALYSIS COMPLETED
Primary SHAP model: Voting Classifier
Results saved to: C:\Users\hosse\Desktop\Baqiyatallah Hospital\Projects\Dr. Mahdi Sotoudeh\Serum Ferritin, D-Dimer, LDH, and CRP as Predictors of Mortality in COVID-19 A Machine Learning Approach\Results_Revised
Figures saved to: C:\Users\hosse\Desktop\Baqiyatallah Hospital\Projects\Dr. Mahdi Sotoudeh\Serum Ferritin, D-Dimer, LDH, and CRP as Predictors of Mortality in COVID-19 A Machine Learning Approach\Results_Revised\Figures
SHAP outputs saved to: C:\Users\hosse\Desktop\Baqiyatallah Hospital\Projects\Dr. Mahdi Sotoudeh\Serum Ferritin, D-Dimer, LDH, and CRP as Predictors of Mortality in COVID-19 A Machine Learning Approach\Results_Revised\Figures\SHAP
Models saved to: C:\Users\hosse\Desktop\Baqiyatallah Hospital\Projects\Dr. Mahdi Sotoudeh\Serum Ferritin, D-Dimer, LDH, and CRP as Predictors of Mortality in COVID-19 A Machine Learning Approach\Results_Revised\Models
